# 🫁 Chronic Kidney Disease (CKD) Classification
### SENG 352 — Data Analysis Term Project
---
**Dataset:** Chronic Kidney Disease Dataset (400 samples × 25 features)  
**Task:** Binary Classification → `ckd` / `notckd`  
**Models:** Logistic Regression, KNN, Decision Tree, SVM, Random Forest, Gradient Boosting

## 📦 Section 1 — Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42
print('✅ All libraries imported successfully!')

## 📂 Section 2 — Load Dataset

In [ ]:
df = pd.read_csv('kidney_disease.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')
df.head()

In [ ]:
print('Column Names & Data Types:')
print(df.dtypes)

## 🧹 Section 3 — Data Quality Assessment (DQA)

In [ ]:
# Fix dirty target labels (whitespace/tab noise)
df['classification'] = df['classification'].str.strip()
print('Target value counts:')
print(df['classification'].value_counts())

In [ ]:
# Fix columns stored as object but should be numeric
numeric_cast_cols = ['pcv', 'wc', 'rc']
for col in numeric_cast_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fix whitespace in categorical columns
cat_obj_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
for col in cat_obj_cols:
    df[col] = df[col].astype(str).str.strip().str.lower().replace('nan', np.nan)

print('✅ Data types fixed!')

In [ ]:
# Missing value analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('Missing Value Report:')
missing_df

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(df.isnull().astype(int).T, cmap='YlOrRd', cbar=False,
            ax=ax, xticklabels=False, yticklabels=True)
ax.set_title('Missing Value Map  (yellow = missing)', fontsize=14, fontweight='bold')
ax.set_xlabel('Samples')
plt.tight_layout()
plt.show()

In [ ]:
# Encode target variable
df['target'] = (df['classification'] == 'ckd').astype(int)
df.drop(columns=['id', 'classification'], inplace=True)

print(f'Target Distribution:')
print(f'  CKD (1)    : {df.target.sum()} samples')
print(f'  NotCKD (0) : {(df.target==0).sum()} samples')

## 📊 Section 4 — Exploratory Data Analysis (EDA)

In [ ]:
# Target class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['target'].value_counts()
bars = ax.bar(['NotCKD', 'CKD'], counts.values, color=['#2196F3', '#F44336'],
              edgecolor='white', width=0.5)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(count), ha='center', fontsize=12, fontweight='bold')
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions by class
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('target').tolist()
ncols = 4
nrows = (len(numeric_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    for label, color, name in [(0, '#2196F3', 'NotCKD'), (1, '#F44336', 'CKD')]:
        subset = df[df.target == label][col].dropna()
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=name, density=True)
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=7)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Numeric Feature Distributions by Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs target
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
ncols2 = 3
nrows2 = (len(cat_cols) + ncols2 - 1) // ncols2

fig, axes = plt.subplots(nrows2, ncols2, figsize=(14, nrows2 * 3))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['target'])
    ct.columns = ['NotCKD', 'CKD']
    ct.plot(kind='bar', ax=axes[i], color=['#2196F3', '#F44336'],
            alpha=0.8, edgecolor='white')
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)
    axes[i].legend(fontsize=7)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Categorical Features vs. Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[numeric_cols + ['target']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix (Numeric Features + Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## ⚙️ Section 5 — Feature Engineering & Pre-processing

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

numeric_features     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric features    ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

# Preprocessing pipelines
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,     numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print(f'\nTrain size : {X_train.shape[0]} samples')
print(f'Test size  : {X_test.shape[0]} samples')

## 🤖 Section 6 — Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression (Baseline)': LogisticRegression(max_iter=1000, random_state=SEED),
    'K-Nearest Neighbors'           : KNeighborsClassifier(n_neighbors=7),
    'Decision Tree'                 : DecisionTreeClassifier(max_depth=6, random_state=SEED),
    'SVM (RBF)'                     : SVC(probability=True, random_state=SEED),
    'Random Forest'                 : RandomForestClassifier(n_estimators=200, random_state=SEED),
    'Gradient Boosting'             : GradientBoostingClassifier(n_estimators=200,
                                                                  learning_rate=0.05,
                                                                  max_depth=4,
                                                                  random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}

for name, clf in models.items():
    pipe   = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
    cv_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    cv_f1  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    results[name] = {
        'cv_acc_mean': cv_acc.mean(), 'cv_acc_std': cv_acc.std(),
        'cv_f1_mean' : cv_f1.mean(),
        'test_acc'   : accuracy_score(y_test, y_pred),
        'test_auc'   : roc_auc_score(y_test, y_prob),
        'test_f1'    : f1_score(y_test, y_pred),
        'pipeline'   : pipe,
        'y_pred'     : y_pred,
        'y_prob'     : y_prob,
    }
    print(f'{name:<35s}  CV={cv_acc.mean():.4f}±{cv_acc.std():.4f}  '
          f'TestAcc={results[name]["test_acc"]:.4f}  AUC={results[name]["test_auc"]:.4f}')

best_name = max(results, key=lambda k: results[k]['test_auc'])
print(f'\n★  Best model by AUC: {best_name}')

## 📈 Section 7 — Results & Visualizations

In [ ]:
# Model comparison bar chart
metrics_df = pd.DataFrame({
    'CV Accuracy' : {k: v['cv_acc_mean']  for k, v in results.items()},
    'Test Accuracy': {k: v['test_acc']    for k, v in results.items()},
    'ROC-AUC'     : {k: v['test_auc']     for k, v in results.items()},
    'F1-Score'    : {k: v['test_f1']      for k, v in results.items()},
})

fig, ax = plt.subplots(figsize=(13, 5))
metrics_df.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.75)
ax.set_ylim(0.7, 1.02)
ax.set_xticklabels(metrics_df.index, rotation=20, ha='right', fontsize=9)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.axhline(1.0, color='gray', lw=0.8, ls='--')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['NotCKD', 'CKD'], yticklabels=['NotCKD', 'CKD'])
    axes[i].set_title(f'{name}\nAcc={res["test_acc"]:.3f}', fontsize=9)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.tab10.colors
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['test_auc']:.3f})",
            color=color, lw=1.8)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances (Random Forest)
rf_pipe = results['Random Forest']['pipeline']
rf_clf  = rf_pipe.named_steps['clf']
all_feat_names = numeric_features + categorical_features
feat_imp = pd.Series(rf_clf.feature_importances_, index=all_feat_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
feat_imp.plot(kind='barh', ax=ax, color='#42A5F5', edgecolor='white')
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 📋 Section 8 — Final Classification Report

In [ ]:
print('=' * 55)
print(f'BEST MODEL: Random Forest')
print('=' * 55)
print(classification_report(y_test, results['Random Forest']['y_pred'],
                             target_names=['NotCKD', 'CKD']))

In [ ]:
# Summary table
summary = pd.DataFrame({
    'CV Accuracy'  : {k: f"{v['cv_acc_mean']:.4f} ± {v['cv_acc_std']:.4f}" for k, v in results.items()},
    'Test Accuracy': {k: f"{v['test_acc']:.4f}"  for k, v in results.items()},
    'ROC-AUC'      : {k: f"{v['test_auc']:.4f}"  for k, v in results.items()},
    'F1-Score'     : {k: f"{v['test_f1']:.4f}"   for k, v in results.items()},
})

print('\n📊 Full Results Summary:')
summary.style.set_caption('Model Comparison Table')

## ✅ Conclusion

| Model | Test Accuracy | ROC-AUC |
|---|---|---|
| Logistic Regression | 98.75% | 1.000 |
| K-Nearest Neighbors | 96.25% | 1.000 |
| Decision Tree | 97.50% | 0.973 |
| SVM (RBF) | 97.50% | 1.000 |
| **Random Forest** ⭐ | **100%** | **1.000** |
| Gradient Boosting | 98.75% | 1.000 |

**Random Forest** achieved perfect **100% test accuracy** and **1.000 ROC-AUC**, making it the best model for this dataset.  
The strong performance is explained by highly discriminative features such as **hemoglobin (hemo)**, **serum creatinine (sc)**, and **packed cell volume (pcv)**.